# YOLOv8n Baseline Training (No CB Loss)

Standart YOLOv8n training without Class-Balanced Loss.  
Class-Balanced Loss kullanmadan standart YOLOv8n egitimi.

All other hyperparameters are kept identical to the CB Loss runs for fair comparison.  
Adil karsilastirma icin diger tum hyperparametreler CB Loss egitimleriyle ayni tutuldu.

## 1. Setup / Kurulum

In [11]:
# Mount Google Drive.
# Google Drive'i baglar.
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
# Install dependencies.
# Bagimliliklari kur.
!pip install -q ultralytics openpyxl pyyaml matplotlib numpy

In [13]:
# Verify GPU availability.
# GPU varligini dogrula.
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


## 2. Configuration / Konfigurasyon

In [14]:
from pathlib import Path

# Dataset path on Drive.
# Drive'daki dataset yolu.
DATA_YAML = "/content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo/data.yaml"

# Output directory on Drive (all artifacts go here).
# Drive'daki cikti dizini (tum cikti bunun icine yazilir).
RUNS_DIR = Path("/content/drive/MyDrive/sayzek_runs_yolov11")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# Run name and Excel file name.
# Run ismi ve Excel dosya ismi.
RUN_NAME    = "yolov8n_baseline_no_cb_augmented_26.04.2026"
OUTPUT_XLSX = "training_metrics.xlsx"

# Training configuration. Identical to CB Loss runs except no CB.
# Egitim konfigurasyonu. CB Loss egitimleriyle ayni, sadece CB yok.
CFG = {
    "IMG_SIZE":       640,
    "DEVICE":         0,
    "EPOCHS":         300,
    "BATCH_SIZE":     32,
    "PATIENCE":       30,
    "WORKERS":        4,
    "CACHE":          False,
    "SEED":           0,

    "OPTIMIZER":      "SGD",
    "LR0":            0.01,
    "LRF":            0.1,
    "MOMENTUM":       0.937,
    "WEIGHT_DECAY":   0.0005,
    "WARMUP_EPOCHS":  3,
    "AMP":            True,

    "HSV_H":          0.015,
    "HSV_S":          0.7,
    "HSV_V":          0.4,
    "MOSAIC":         0.0,
    "MIXUP":          0.0,
    "FLIPUD":         0.0,
    "FLIPLR":         0.0,
    "DEGREES":        0.0,
    "TRANSLATE":      0.0,
    "SCALE":          0.0,
    "PERSPECTIVE":    0.0,
    "ERASING":        0.0,
}

In [15]:
# Fix data.yaml paths to match the actual dataset location.
# data.yaml yollarini gercek dataset konumuna gore duzelt.
#
# The shared data.yaml has 'path' pointing to the original owner's location.
# Paylasilan data.yaml'in 'path' alani orijinal sahibin konumunu gosteriyor.
# We rewrite a local copy with the correct path.
# Dogru path ile lokal bir kopya olusturuyoruz.
import yaml

DATASET_ROOT = "/content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo"
DATA_YAML = f"{DATASET_ROOT}/data.yaml"

with open(DATA_YAML, "r") as f:
    data_yaml_content = yaml.safe_load(f)

data_yaml_content["path"]  = DATASET_ROOT
data_yaml_content["train"] = "images/train"
data_yaml_content["val"]   = "images/val"
data_yaml_content["test"]  = "images/test"

FIXED_DATA_YAML = "/content/data_fixed.yaml"
with open(FIXED_DATA_YAML, "w") as f:
    yaml.safe_dump(data_yaml_content, f, sort_keys=False)

print("Fixed data.yaml content:")
print(yaml.safe_dump(data_yaml_content, sort_keys=False))

# Switch to the fixed yaml for the rest of the notebook.
# Notebook'un geri kalani icin duzeltilmis yaml'a gec.
DATA_YAML = FIXED_DATA_YAML


Fixed data.yaml content:
path: /content/drive/MyDrive/Sayzek/dataset_augmented_yolo/dataset_augmented_yolo
train: images/train
val: images/val
test: images/test
nc: 3
names:
  0: car
  1: person
  2: other_vehicle



## 3. Read Class Names from data.yaml / data.yaml'dan sinif isimlerini oku

In [16]:
import yaml

def read_class_names(data_yaml: str) -> list:
    # Load class names from the data.yaml file.
    # data.yaml dosyasindan sinif isimlerini yukler.
    with open(data_yaml, "r") as f:
        data = yaml.safe_load(f)
    names = data.get("names")
    if names is None:
        raise ValueError(f"'names' field not found in {data_yaml}")
    if isinstance(names, dict):
        return [names[k] for k in sorted(names.keys())]
    elif isinstance(names, list):
        return list(names)
    else:
        raise ValueError(f"'names' must be list or dict, got: {type(names)}")

CLASS_NAMES = read_class_names(DATA_YAML)
print(f"Classes / Siniflar: {CLASS_NAMES}")

Classes / Siniflar: ['car', 'person', 'other_vehicle']


## 4. Excel Logging Helpers / Excel kayit yardimcilari

In [17]:
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from typing import Optional

STATIC_HEADERS = [
    "Epoch", "Learning Rate",
    "Box Loss (Train)", "Box Loss (Val)",
    "Class Loss (Train)", "Class Loss (Val)",
    "DFL Loss (Train)", "DFL Loss (Val)",
    "Precision (Val)", "Recall (Val)", "F1 (Val)",
    "mAP@0.5 (overall)", "mAP@0.5:0.95 (overall)",
]

_THIN        = Side(border_style="thin", color="BBBBBB")
BORDER       = Border(left=_THIN, right=_THIN, top=_THIN, bottom=_THIN)
HEADER_FILL  = PatternFill("solid", start_color="1F3864")
HEADER_FONT  = Font(name="Arial", bold=True, color="FFFFFF", size=10)
DATA_FONT    = Font(name="Arial", size=10)


def build_headers(class_names: list) -> list:
    # Build the full header row based on class names.
    # Sinif isimlerine gore tam header satirini olusturur.
    headers = list(STATIC_HEADERS)
    for name in class_names:
        headers.append(f"mAP@0.5 ({name})")
        headers.append(f"mAP@0.5:0.95 ({name})")
        headers.append(f"Precision ({name})")
        headers.append(f"Recall ({name})")
    return headers


def apply_header(ws, class_names: list) -> None:
    # Apply single-row header to the Training Metrics sheet.
    # Training Metrics sayfasina tek satirli header uygular.
    headers = build_headers(class_names)
    for ci, val in enumerate(headers, 1):
        c = ws.cell(row=1, column=ci, value=val)
        c.fill = HEADER_FILL
        c.font = HEADER_FONT
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border = BORDER
    for i in range(1, len(headers) + 1):
        width = 10 if i <= len(STATIC_HEADERS) else 18
        ws.column_dimensions[get_column_letter(i)].width = width
    ws.row_dimensions[1].height = 40
    ws.freeze_panes = "A2"


def build_excel(output_path: Path, class_names: list) -> None:
    # Create a new Excel file with three sheets.
    # Uc sayfali yeni bir Excel dosyasi olusturur.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    wb = openpyxl.Workbook()

    ws = wb.active
    ws.title = "Training Metrics"
    apply_header(ws, class_names)

    cm = wb.create_sheet("Confusion Matrix")
    cm["A1"] = "Confusion Matrix - Egitim tamamlandiktan sonra doldurulur."
    cm["A1"].font = Font(name="Arial", bold=True, size=11)

    ch = wb.create_sheet("Charts")
    for r, txt in enumerate([
        "Grafikler bu sayfaya eklenecektir.",
        "Insert -> Chart ile Training Metrics verilerini kullanabilirsiniz.",
    ], start=1):
        c = ch.cell(row=r, column=1, value=txt)
        c.font = Font(name="Arial", size=10, italic=True, color="555555")

    wb.save(output_path)
    print(f"Excel created: {output_path}")


def fmt(v) -> Optional[float]:
    # Preserve None, round float to 4 decimals.
    # None'i korur, float degerini 4 ondalikla yuvarlar.
    return None if v is None else round(float(v), 4)


def append_epoch_row(output_path: Path, row_values: list) -> None:
    # Append a single epoch row to the existing Excel file.
    # Mevcut Excel dosyasina tek bir epoch satiri ekler.
    wb = openpyxl.load_workbook(output_path)
    ws = wb["Training Metrics"]
    data_row = ws.max_row + 1
    alt_fill = (PatternFill("solid", start_color="EBF3FB")
                if data_row % 2 == 0 else None)
    for ci, val in enumerate(row_values, 1):
        c = ws.cell(row=data_row, column=ci, value=val)
        c.font = DATA_FONT
        c.alignment = Alignment(horizontal="center")
        c.border = BORDER
        if alt_fill:
            c.fill = alt_fill
    wb.save(output_path)

## 5. Training Callback / Egitim callback'i

Per-class metrics are collected from `trainer.validator.metrics.box`.  
Per-class metrikler `trainer.validator.metrics.box`'tan toplanir.

No CB Loss injection here, just metric logging.  
Burada CB Loss inject islemi yok, sadece metrik kaydi var.

In [18]:
from ultralytics.utils import LOGGER

class BaselineCallbacks:
    # Callback class for baseline training (no CB Loss).
    # Baseline egitimi icin callback sinifi (CB Loss yok).

    def __init__(self, output_xlsx_name: str, class_names: list):
        self.output_xlsx_name = output_xlsx_name
        self.class_names      = class_names
        self.output_path      = None

    def on_pretrain_routine_end(self, trainer) -> None:
        # Use the actual YOLO save_dir so Excel lands next to other artifacts.
        # Excel'in diger ciktilarla ayni yere dusmesi icin gercek save_dir'i kullan.
        save_dir = Path(trainer.save_dir)
        self.output_path = save_dir / self.output_xlsx_name
        if not self.output_path.exists():
            build_excel(self.output_path, self.class_names)
        LOGGER.info(f"[Baseline] Excel will be written to: {self.output_path}")

    def on_fit_epoch_end(self, trainer) -> None:
        # Collect metrics at the end of an epoch and write them to Excel.
        # Epoch sonunda metrikleri toplar ve Excel'e yazar.
        epoch      = trainer.epoch
        metrics    = trainer.metrics
        loss_items = trainer.loss_items
        lr_list    = trainer.scheduler.get_last_lr()
        lr_val     = lr_list[0] if lr_list else trainer.args.lr0

        def _li(i):
            return float(loss_items[i]) if loss_items is not None else None

        box_t, cls_t, dfl_t = _li(0), _li(1), _li(2)

        def _m(key):
            return float(metrics[key]) if key in metrics else None

        box_v    = _m("val/box_loss")
        cls_v    = _m("val/cls_loss")
        dfl_v    = _m("val/dfl_loss")
        prec     = _m("metrics/precision(B)")
        rec      = _m("metrics/recall(B)")
        map50    = _m("metrics/mAP50(B)")
        map50_95 = _m("metrics/mAP50-95(B)")

        f1 = None
        if prec is not None and rec is not None:
            denom = prec + rec
            f1 = (2 * prec * rec / denom) if denom > 0 else 0.0

        per_class = self._collect_per_class(trainer)

        row = [
            epoch, round(lr_val, 8),
            fmt(box_t), fmt(box_v),
            fmt(cls_t), fmt(cls_v),
            fmt(dfl_t), fmt(dfl_v),
            fmt(prec), fmt(rec), fmt(f1),
            fmt(map50), fmt(map50_95),
        ]
        for i in range(len(self.class_names)):
            row.extend([
                fmt(per_class["map50"][i]),
                fmt(per_class["map50_95"][i]),
                fmt(per_class["precision"][i]),
                fmt(per_class["recall"][i]),
            ])

        append_epoch_row(self.output_path, row)

        if all(v is not None for v in [box_t, cls_t, dfl_t, map50]):
            LOGGER.info(
                f"[Baseline] Epoch {epoch:3d} -> "
                f"box={box_t:.4f}  cls={cls_t:.4f}  dfl={dfl_t:.4f}  "
                f"mAP@0.5={map50:.4f}"
            )

    def _collect_per_class(self, trainer) -> dict:
        # Collect per-class metrics from trainer.validator.metrics.box.
        # trainer.validator.metrics.box'tan per-class metrikleri toplar.
        nc = len(self.class_names)
        result = {
            "map50":     [None] * nc,
            "map50_95":  [None] * nc,
            "precision": [None] * nc,
            "recall":    [None] * nc,
        }
        validator = getattr(trainer, "validator", None)
        if validator is None:
            return result
        v_metrics = getattr(validator, "metrics", None)
        if v_metrics is None:
            return result
        box = getattr(v_metrics, "box", None)
        if box is None:
            return result
        ap_class_index = getattr(box, "ap_class_index", None)
        if ap_class_index is None or len(ap_class_index) == 0:
            return result
        maps     = getattr(box, "maps", None)
        p_arr    = getattr(box, "p", None)
        r_arr    = getattr(box, "recall", None)
        if r_arr is None:
            r_arr = getattr(box, "r", None)
        ap50_arr = getattr(box, "ap50", None)
        for pos, cls_idx in enumerate(ap_class_index):
            cls_idx = int(cls_idx)
            if cls_idx >= nc:
                continue
            if ap50_arr is not None and pos < len(ap50_arr):
                result["map50"][cls_idx] = ap50_arr[pos]
            if maps is not None and cls_idx < len(maps):
                result["map50_95"][cls_idx] = maps[cls_idx]
            if p_arr is not None and pos < len(p_arr):
                result["precision"][cls_idx] = p_arr[pos]
            if r_arr is not None and pos < len(r_arr):
                result["recall"][cls_idx] = r_arr[pos]
        return result

    def on_train_end(self, trainer) -> None:
        LOGGER.info(f"[Baseline] Training finished. Output: {self.output_path}")

## 6. Start Training / Egitimi Baslat

In [19]:
from ultralytics import YOLO

print(f"Run name:    {RUN_NAME}")
print(f"Output dir:  {RUNS_DIR}")
print(f"Excel name:  {OUTPUT_XLSX}")
print(f"Classes:     {CLASS_NAMES}")
print(f"Seed:        {CFG['SEED']}")
print()

# Load YOLOv11n base model.
# YOLOv11n temel modelini yukle.
model = YOLO("yolo11n.pt")

# Attach baseline callbacks.
# Baseline callback'lerini bagla.
cbs = BaselineCallbacks(
    output_xlsx_name=OUTPUT_XLSX,
    class_names=CLASS_NAMES,
)
model.add_callback("on_pretrain_routine_end", cbs.on_pretrain_routine_end)
model.add_callback("on_fit_epoch_end",        cbs.on_fit_epoch_end)
model.add_callback("on_train_end",            cbs.on_train_end)

# Run training.
# Egitimi calistir.
model.train(
    data=DATA_YAML,
    imgsz=CFG["IMG_SIZE"],
    device=CFG["DEVICE"],
    epochs=CFG["EPOCHS"],
    batch=CFG["BATCH_SIZE"],
    patience=CFG["PATIENCE"],
    workers=CFG["WORKERS"],
    cache=CFG["CACHE"],
    seed=CFG["SEED"],
    optimizer=CFG["OPTIMIZER"],
    lr0=CFG["LR0"],
    lrf=CFG["LRF"],
    momentum=CFG["MOMENTUM"],
    weight_decay=CFG["WEIGHT_DECAY"],
    warmup_epochs=CFG["WARMUP_EPOCHS"],
    amp=CFG["AMP"],
    hsv_h=CFG["HSV_H"],
    hsv_s=CFG["HSV_S"],
    hsv_v=CFG["HSV_V"],
    mosaic=CFG["MOSAIC"],
    mixup=CFG["MIXUP"],
    flipud=CFG["FLIPUD"],
    fliplr=CFG["FLIPLR"],
    degrees=CFG["DEGREES"],
    translate=CFG["TRANSLATE"],
    scale=CFG["SCALE"],
    perspective=CFG["PERSPECTIVE"],
    erasing=CFG["ERASING"],
    project=str(RUNS_DIR),
    name=RUN_NAME,
    verbose=True,
)


Run name:    yolov8n_baseline_no_cb_augmented_26.04.2026
Output dir:  /content/drive/MyDrive/sayzek_runs_yolov11
Excel name:  training_metrics.xlsx
Classes:     ['car', 'person', 'other_vehicle']
Seed:        0

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.0, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.1, mas

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a268e6823c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.04

## 7. Generate Plots / Grafikleri Uret

In [20]:
# Generate plots / Grafikleri uret
#
# Reads the Excel file we wrote during training and generates loss/metric plots
# next to it (under the YOLO run directory on Drive).
# Egitim sirasinda yazdigimiz Excel dosyasini okur ve grafikleri yaninda
# (Drive'daki YOLO run dizini altinda) olusturur.

import os
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import openpyxl

# Style / Stil
LW = 2.0
CLASS_COLORS = ["#1F77B4", "#FF7F0E", "#2CA02C", "#D62728", "#9467BD",
                "#8C564B", "#E377C2", "#7F7F7F", "#BCBD22", "#17BECF"]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.color": "#e0e0e0",
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#cccccc",
})


# Configuration / Konfigurasyon
SKIP = 3
DPI = 150
FMT = "png"
PER_CLASS = True


# Auto-detect the actual run directory.
# Gercek run dizinini otomatik tespit et.
# YOLO may append a suffix like -2, -3 if the folder already existed.
# YOLO klasor zaten varsa -2, -3 gibi son ek ekleyebilir.

def find_run_dir(runs_dir, base_name):
    candidates = []
    for item in os.listdir(runs_dir):
        full = runs_dir / item
        if full.is_dir() and item.startswith(base_name):
            xlsx = full / OUTPUT_XLSX
            if xlsx.exists():
                candidates.append((full, xlsx.stat().st_mtime))
    if not candidates:
        raise FileNotFoundError(
            f"No directory starting with '{base_name}' containing "
            f"'{OUTPUT_XLSX}' was found under {runs_dir}"
        )
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]


RUN_DIR    = find_run_dir(RUNS_DIR, RUN_NAME)
EXCEL_PATH = RUN_DIR / OUTPUT_XLSX
PLOTS_DIR  = RUN_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Detected run dir: {RUN_DIR.name}")
print(f"Reading: {EXCEL_PATH}")
print(f"Output:  {PLOTS_DIR}")
print()


# Load Excel data / Excel verisini yukle
def to_float_array(values):
    out = []
    for v in values:
        if v is None or v == "":
            out.append(np.nan)
        else:
            try:
                out.append(float(v))
            except (TypeError, ValueError):
                out.append(np.nan)
    return np.array(out, dtype=np.float64)


wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb["Training Metrics"]
rows = list(ws.iter_rows(values_only=True))
headers = [str(h).strip() if h is not None else "" for h in rows[0]]
data_rows = rows[1:]


def col(name):
    idx = headers.index(name)
    return to_float_array([r[idx] for r in data_rows])


d = {
    "epoch":   col("Epoch"),
    "t_box":   col("Box Loss (Train)"),
    "v_box":   col("Box Loss (Val)"),
    "t_cls":   col("Class Loss (Train)"),
    "v_cls":   col("Class Loss (Val)"),
    "t_dfl":   col("DFL Loss (Train)"),
    "v_dfl":   col("DFL Loss (Val)"),
    "prec":    col("Precision (Val)"),
    "rec":     col("Recall (Val)"),
    "map50":   col("mAP@0.5 (overall)"),
    "map5095": col("mAP@0.5:0.95 (overall)"),
}

prefix = "mAP@0.5 ("
class_names_from_excel = []
for h in headers:
    if h.startswith(prefix) and h.endswith(")") and h != "mAP@0.5 (overall)":
        class_names_from_excel.append(h[len(prefix):-1])

per_class = {}
for name in class_names_from_excel:
    per_class[name] = {
        "map50":     col(f"mAP@0.5 ({name})"),
        "map5095":   col(f"mAP@0.5:0.95 ({name})"),
        "precision": col(f"Precision ({name})"),
        "recall":    col(f"Recall ({name})"),
    }
d["class_names"] = class_names_from_excel
d["per_class"]   = per_class

mask = d["epoch"] > SKIP
for k in list(d.keys()):
    if k == "class_names":
        continue
    if k == "per_class":
        d[k] = {n: {mk: mv[mask] for mk, mv in m.items()} for n, m in d[k].items()}
    else:
        d[k] = d[k][mask]

print(f"Total epochs (after skip {SKIP}): {len(d['epoch'])}")
print(f"Classes: {d['class_names']}")
print()


def style_ax(ax, ep, title, ylabel="Loss"):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    if len(ep) > 0:
        ax.set_xlim(ep[0], ep[-1])
    ax.legend()


# Plot 1: Box Loss
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d["epoch"], d["t_box"], color="#2166AC", linewidth=LW, label="Train Loss")
ax.plot(d["epoch"], d["v_box"], color="#D73027", linewidth=LW, linestyle="--", label="Validation Loss")
style_ax(ax, d["epoch"], "Box Loss")
plt.tight_layout()
path = PLOTS_DIR / f"loss_box.{FMT}"
fig.savefig(path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {path}")

# Plot 2: Class Loss
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d["epoch"], d["t_cls"], color="#D73027", linewidth=LW, label="Train Loss")
ax.plot(d["epoch"], d["v_cls"], color="#FC8D59", linewidth=LW, linestyle="--", label="Validation Loss")
style_ax(ax, d["epoch"], "Class Loss")
plt.tight_layout()
path = PLOTS_DIR / f"loss_cls.{FMT}"
fig.savefig(path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {path}")

# Plot 3: DFL Loss
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(d["epoch"], d["t_dfl"], color="#1A9850", linewidth=LW, label="Train Loss")
ax.plot(d["epoch"], d["v_dfl"], color="#74C476", linewidth=LW, linestyle="--", label="Validation Loss")
style_ax(ax, d["epoch"], "DFL Loss")
plt.tight_layout()
path = PLOTS_DIR / f"loss_dfl.{FMT}"
fig.savefig(path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {path}")

# Plot 4: Three combined
ep = d["epoch"]
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(ep, d["t_box"], color="#2166AC", linewidth=LW, label="Box - Train")
ax.plot(ep, d["v_box"], color="#2166AC", linewidth=LW, linestyle="--", label="Box - Val")
ax.plot(ep, d["t_cls"], color="#D73027", linewidth=LW, label="Cls - Train")
ax.plot(ep, d["v_cls"], color="#D73027", linewidth=LW, linestyle="--", label="Cls - Val")
ax.plot(ep, d["t_dfl"], color="#1A9850", linewidth=LW, label="DFL - Train")
ax.plot(ep, d["v_dfl"], color="#1A9850", linewidth=LW, linestyle="--", label="DFL - Val")
ax.set_title("Box + Class + DFL Loss", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
if len(ep) > 0:
    ax.set_xlim(ep[0], ep[-1])
ax.legend(ncol=3, loc="upper right")
plt.tight_layout()
path = PLOTS_DIR / f"loss_three_combined.{FMT}"
fig.savefig(path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {path}")

# Plot 5: Combined dashboard
fig = plt.figure(figsize=(18, 10))
fig.suptitle("Training Metrics / Egitim Metrikleri", fontsize=16, fontweight="bold", y=1.01)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)

for col_idx, (key_t, key_v, title, tc, vc) in enumerate([
    ("t_box", "v_box", "Box Loss",   "#2166AC", "#D73027"),
    ("t_cls", "v_cls", "Class Loss", "#D73027", "#FC8D59"),
    ("t_dfl", "v_dfl", "DFL Loss",   "#1A9850", "#74C476"),
]):
    ax = fig.add_subplot(gs[0, col_idx])
    ax.plot(ep, d[key_t], color=tc, linewidth=LW, label="Train Loss")
    ax.plot(ep, d[key_v], color=vc, linewidth=LW, linestyle="--", label="Validation Loss")
    style_ax(ax, ep, title)

ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(ep, d["prec"], color="#2166AC", linewidth=LW, label="Validation Precision")
style_ax(ax4, ep, "Precision", ylabel="Precision")

ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(ep, d["rec"], color="#2166AC", linewidth=LW, label="Validation Recall")
style_ax(ax5, ep, "Recall", ylabel="Recall")

ax6 = fig.add_subplot(gs[1, 2])
ax6.plot(ep, d["map50"],   color="#2166AC", linewidth=LW, label="mAP@0.5")
ax6.plot(ep, d["map5095"], color="#D73027", linewidth=LW, linestyle="--", label="mAP@0.5:0.95")
style_ax(ax6, ep, "mAP", ylabel="mAP")

path = PLOTS_DIR / f"loss_combined.{FMT}"
fig.savefig(path, dpi=DPI, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {path}")


def save_per_class(metric_key, title, ylabel, filename):
    if not d["class_names"]:
        return
    fig, ax = plt.subplots(figsize=(10, 6))
    for i, name in enumerate(d["class_names"]):
        color = CLASS_COLORS[i % len(CLASS_COLORS)]
        ax.plot(ep, d["per_class"][name][metric_key], color=color, linewidth=LW, label=name)
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    if len(ep) > 0:
        ax.set_xlim(ep[0], ep[-1])
    ax.legend(loc="best")
    plt.tight_layout()
    path = PLOTS_DIR / f"{filename}.{FMT}"
    fig.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {path}")


if PER_CLASS:
    save_per_class("map50",     "Per-Class mAP@0.5",      "mAP@0.5",      "per_class_map50")
    save_per_class("map5095",   "Per-Class mAP@0.5:0.95", "mAP@0.5:0.95", "per_class_map5095")
    save_per_class("precision", "Per-Class Precision",    "Precision",    "per_class_precision")
    save_per_class("recall",    "Per-Class Recall",       "Recall",       "per_class_recall")

print()
print(f"Done. All plots saved to: {PLOTS_DIR}")

Detected run dir: yolov8n_baseline_no_cb_augmented_26.04.2026-3
Reading: /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/training_metrics.xlsx
Output:  /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/plots

Total epochs (after skip 3): 47
Classes: ['car', 'person', 'other_vehicle']

Saved: /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/plots/loss_box.png
Saved: /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/plots/loss_cls.png
Saved: /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/plots/loss_dfl.png
Saved: /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/plots/loss_three_combined.png
Saved: /content/drive/MyDrive/sayzek_runs_yolov11/yolov8n_baseline_no_cb_augmented_26.04.2026-3/plots/loss_combined.png
Saved: /content/drive/MyDrive/sayzek_runs_y

In [21]:
import os

run_dir = str(find_run_dir(RUNS_DIR, RUN_NAME))
print("Klasor icerigi:")
for item in sorted(os.listdir(run_dir)):
    full = os.path.join(run_dir, item)
    if os.path.isdir(full):
        print(f"  [DIR]  {item}/")
        for sub in sorted(os.listdir(full)):
            sub_full = os.path.join(full, sub)
            if os.path.isfile(sub_full):
                size_mb = os.path.getsize(sub_full) / 1024 / 1024
                print(f"           {sub} ({size_mb:.2f} MB)")
    else:
        size_mb = os.path.getsize(full) / 1024 / 1024
        print(f"  [FILE] {item} ({size_mb:.2f} MB)")

Klasor icerigi:
  [FILE] BoxF1_curve.png (0.17 MB)
  [FILE] BoxPR_curve.png (0.14 MB)
  [FILE] BoxP_curve.png (0.16 MB)
  [FILE] BoxR_curve.png (0.16 MB)
  [FILE] args.yaml (0.00 MB)
  [FILE] confusion_matrix.png (0.13 MB)
  [FILE] confusion_matrix_normalized.png (0.13 MB)
  [FILE] labels.jpg (0.11 MB)
  [DIR]  plots/
           loss_box.png (0.09 MB)
           loss_cls.png (0.09 MB)
           loss_combined.png (0.32 MB)
           loss_dfl.png (0.08 MB)
           loss_three_combined.png (0.18 MB)
           per_class_map50.png (0.07 MB)
           per_class_map5095.png (0.09 MB)
           per_class_precision.png (0.12 MB)
           per_class_recall.png (0.09 MB)
  [FILE] results.csv (0.01 MB)
  [FILE] results.png (0.26 MB)
  [FILE] train_batch0.jpg (0.46 MB)
  [FILE] train_batch1.jpg (0.45 MB)
  [FILE] train_batch2.jpg (0.47 MB)
  [FILE] training_metrics.xlsx (0.01 MB)
  [FILE] val_batch0_labels.jpg (0.46 MB)
  [FILE] val_batch0_pred.jpg (0.48 MB)
  [FILE] val_batch1_labels.jpg (

## 8. Done / Bitti

All artifacts (`best.pt`, `last.pt`, `training_metrics.xlsx`, plots, etc.) are saved under the detected YOLO run directory in `/content/drive/MyDrive/sayzek_runs/`.

Tum ciktilar (`best.pt`, `last.pt`, `training_metrics.xlsx`, grafikler, vb.) `/content/drive/MyDrive/sayzek_runs/` altinda tespit edilen YOLO run dizinine kaydedilir.